# First-order logic

_Artificial Intelligence — more_

**Talk about objects, their properties, and 'for all' / 'there exists'. Then reason by unifying and resolving.**

Propositional logic only has whole true/false facts. First-order logic (FOL) can talk about objects and their relations.

     It adds predicates like $P(x,y)$ ("$x$ relates to $y$"), variables, and quantifiers $\forall$ ("for all") and $\exists$ ("there exists").

---

This notebook is a practice scaffold for the **First-order logic** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

### Step 1 — Decide what counts as a variableUnification matches two logical atoms by figuring out which variables must stand for which terms. The first thing the algorithm needs is a way to tell a **variable** apart from a constant. Here the convention is simple: a lowercase-initial string (like `x` or `y`) is a variable, while anything else (like `John` or `Alice`) is a fixed constant.

In [ ]:
def is_var(t):
    # Lowercase-initial strings are variables; everything else is a constant.
    return isinstance(t, str) and t[0].islower()

### Step 2 — Unify two terms, recursively`unify` walks two terms in parallel and builds up a **substitution** `s` (a dict mapping each variable to the term it stands for). Identical terms need nothing. If either side is a variable, we bind it. If both sides are tuples of the same length (e.g. whole atoms like `("Knows", "x", "John")`), we unify them position by position, threading the growing substitution through. Anything else is a clash, so we return `None`.

In [ ]:
def unify(a, b, s=None):
    # s is the substitution built so far; start it empty on the first call.
    if s is None:
        s = {}

    if a == b:
        # Identical terms are already unified; nothing to add.
        return s

    if is_var(a):
        return unify_var(a, b, s)

    if is_var(b):
        return unify_var(b, a, s)

    both_tuples = isinstance(a, tuple) and isinstance(b, tuple)
    if both_tuples and len(a) == len(b):
        # Unify the two atoms one position at a time.
        for x, y in zip(a, b):
            s = unify(x, y, s)
            if s is None:
                return None
        return s

    # Mismatched constants or shapes: no unifier exists.
    return None

def unify_var(v, x, s):
    # If v is already bound, unify what it stands for with x instead.
    if v in s:
        return unify(s[v], x, s)

    # Otherwise record the new binding v -> x in a fresh copy of s.
    s = dict(s)
    s[v] = x
    return s

### Step 3 — Try it on two atomsNow we unify `Knows(x, John)` with `Knows(Alice, y)`. Matching position by position forces `x = Alice` and `y = John`, so the substitution makes both atoms identical. The second example, `P(Alice)` vs `P(Bob)`, has two clashing constants and no variables to absorb the difference, so unification fails and returns `None`.

In [ ]:
# Knows(x, John) vs Knows(Alice, y)
atom1 = ("Knows", "x", "John")
atom2 = ("Knows", "Alice", "y")

print("unifier:", unify(atom1, atom2))            # {x: Alice, y: John}
print("no match:", unify(("P", "Alice"), ("P", "Bob")))

## Visualize the data & results

_Family knowledge base: applying the grandparent rule by forward chaining, how many facts can we derive?_

### Step 1 — Assert the base factsWe start from a small family knowledge base: a list of `parent` pairs. Each pair becomes a `Parent(...)` fact stored in a set. Counting them now gives us the "before" number — the facts we know **before** any reasoning is applied.

In [ ]:
import matplotlib.pyplot as plt

# Family relations knowledge base: each pair is (parent, child).
parent = [("Tom", "Bob"), ("Bob", "Ann"), ("Tom", "Liz"), ("Liz", "Pat")]

# Turn each parent pair into a Parent(...) fact.
facts = set(("Parent",) + p for p in parent)
before = len(facts)

### Step 2 — Forward-chain the grandparent ruleForward chaining applies a rule to known facts to derive new ones. The rule here is: `Parent(x, y) AND Parent(y, z) => Grandparent(x, z)`. We look for every pair of parent links where the child of the first is the parent of the second — that shared middle person `y` is what connects a grandparent to a grandchild. Each match adds a new `Grandparent(...)` fact, which we then merge back into the knowledge base.

In [ ]:
# Apply Parent(x,y) AND Parent(y,z) => Grandparent(x,z).
grand = set()
for (x, y) in parent:
    for (y2, z) in parent:
        if y == y2:                       # the shared middle person links the two
            grand.add(("Grandparent", x, z))

facts |= grand                            # add the derived facts to the KB
after = len(facts)

print("derived:", sorted(g[1] + "-" + g[2] for g in grand))  # Tom-Ann, Tom-Pat

### Step 3 — Plot how many facts we knowFinally we compare the size of the knowledge base before and after forward chaining. The bar chart makes the gain visible: the grandparent rule turned the asserted parent facts into a strictly larger set of known facts, all derived automatically.

In [ ]:
labels = ["asserted Parent facts", "after grandparent rule"]
counts = [before, after]

fig, ax = plt.subplots()
bars = ax.bar(labels, counts, color=["#ffb454", "#7ee787"])

# Label each bar with its count.
for bar, v in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, v, str(v), ha="center", va="bottom")

ax.set_ylabel("facts known")
ax.set_title("Facts in the family KB: asserted vs after forward chaining")
plt.show()